# 03 — Phase 1b: PF Exercise Generation — Brainstorming

Open questions, design decisions, and what we learned from Phase 1a.

## What we have

- Phase 1a pipeline (MCQ distractors — working end-to-end)
- 200 Eedi questions with misconception labels
- 15 expert-authored PF exercises from StratL
- Rahimi et al. (2025) paper — PF rubric + GPT-4o baseline (13.38/15)


## Design intuition — what Phase 1a taught us

- **Simpler prompts win.** Over-engineered prompts (CoT, taxonomy chains) underperformed. Phase 1b keeps prompts lean: three versions, one variable (misconception source), nothing else.

- **Reuse proven pipeline.** YAML templates, ThreadPool parallel execution, LLM-as-judge, blinding + shuffling + temp=0, CSV output — all validated in Phase 1a, all transfer directly.



## What goes in? — Defining the Input

### What the papers use as input

| Paper | Input given to generator |
|-------|------------------------|
| **Rahimi et al. (2025)** | Grade level + Common Core standard + theme + 5 human-written examples + PF rubric |
| **Kapur & Bielaczyc (2012)** | Hand-designed by learning scientists — no automated generation |
| **StratL (Puech et al. 2025)** | Existing well-structured problems + pedagogical intent labels — transformed by GPT-4o |

Neither Kapur nor Loibl generated exercises — they designed them manually. Only Rahimi and StratL use LLM generation.

### Our three input versions

Same generator, same model, same system prompt (PF definition + rubric + output format), same 3 few-shot examples from StratL. The only variable is **what the generator knows about student misconceptions**.

| Version | Misconception source | Input fields | Description |
|---|---|---|---|
| **V1 — Blind** | None | Grade + topic + theme | Teacher types the bare minimum. Generator gets no misconception guidance. |
| **V2 — Eedi-grounded** | Real student data | Grade + topic + theme + Eedi misconception list | Generator receives actual misconceptions from the Eedi dataset for that topic. |
| **V3 — LLM-inferred** | LLM guesses them | Grade + topic + theme | Generator first infers likely misconceptions for the topic, then uses those to generate the exercise. A two-step generation: "What do students get wrong about this?" → "Now write the PF exercise." |

### What each comparison tells us

```
V1 ────────── V3 ────────── V2
(blind)    (LLM guesses)   (real data)
```

- **V1 → V3 gap**: Does *any* misconception reasoning help? If V3 > V1, then the generator benefits from thinking about what students get wrong. If no difference, stop here — ship V1.
- **V3 → V2 gap**: Does *real* student data beat what the LLM already knows? If V2 > V3, external misconception data adds value. If V3 ≈ V2, the LLM is good enough — no database needed.

### What this means for the tool

| Outcome | Tool ships with |
|---|---|
| V3 ≈ V2 > V1 | LLM self-infers misconceptions internally. Teacher just types a topic. Simplest interface. |
| V2 > V3 > V1 | Misconception data matters. Tool needs a misconception source: either a built-in knowledge base or a teacher input field. |
| V1 ≈ V3 ≈ V2 | Misconceptions don't move the needle. Ship V1, don't overthink it. |

**Side research question:** Does providing student misconception data improve the quality of LLM-generated PF exercises, as measured by the Rahimi et al. (2025) PF rubric? And is LLM-inferred misconception knowledge as effective as real student data?

**Hypotheses:** (a) V2 and V3 > V1 on SSC and APK — misconception awareness helps rig the data trap and activate relevant prior knowledge. (b) V2 ≈ V3 — LLMs encode enough pedagogical content knowledge to infer reasonable misconceptions without external data. (c) V1 ≈ V2 ≈ V3 on AE and MR — narrative craft and format variety are orthogonal to misconception knowledge.

### The anatomy of a PF exercise

```mermaid
flowchart TD
    A["Narrative Context\nCharacters + stakes + real-world scenario"]
    B["Data / Constraints\nNumbers embedded in story,\ndeliberately rigged"]
    C["Open-Ended Task\n'Design' 'Invent' 'Decide'\nNOT 'solve for X'"]
    D["Critical Conceptual Feature\n(Kapur 2012)\nPrior knowledge -> plausible\nbut WRONG answer"]
    E["RSM Diversity\nDifferent students -> different\nwrong solutions"]
    
    A --> B --> C --> D --> E
    
    style A fill:#2196F3,color:#fff
    style B fill:#4CAF50,color:#fff
    style C fill:#FF9800,color:#fff
    style D fill:#E91E63,color:#fff
    style E fill:#9C27B0,color:#fff
```

### Each element, with examples

| Element | Kapur (2012) | Loibl & Rummel (2014) | Rahimi et al. (2025) |
|---------|-------------|----------------------|---------------------|
| Narrative | Two friends biking to an audition | Soccer players competing for an award | Steph Curry at practice |
| Data | Two bike speeds, two walk speeds, 5 km | Three players, SAME mean, SAME range | 8/20 shots made |
| Task | "How far should she ride so they arrive at the same time?" | "Invent a method to determine who is most consistent" | "Design at least two different ways to solve" |
| Critical conceptual feature | Average speed is total distance / total time, NOT arithmetic mean of speeds (Kapur 2012, p. 1008) | All three have same range AND mean -> need a NEW measure of spread (Loibl & Rummel 2014) | Percentage change is not additive (SSC — Rahimi et al. 2025) |
| RSMs | 9 different student solution attempts observed | Range, mean, visual, neighbor-diff, sum-of-deviations, ... | Multiple correct methods possible |

### Terminology note

The papers do not use the word "trap." Instead:
- **Kapur & Bielaczyc (2012):** `critical_conceptual_features` — the specific conceptual insight that prior knowledge misses
- **Loibl & Rummel (2014):** `design_features` (how the problem is rigged) + `critical_conceptual_features` (what students must discover)
- **Rahimi et al. (2025):** Sweet Spot Calibration (SSC) — the rubric dimension that captures whether the problem is well-calibrated to challenge prior knowledge

The common principle across all papers: a PF exercise must contain a feature where **prior knowledge produces a reasonable-looking but wrong answer**, forcing students to confront the limits of what they know.

## System prompt — defining what the generator produces

This is the fixed prompt that defines the task for the LLM. It stays the same across all three input versions (V1, V2, V3). Only the user-provided fields (grade, topic, theme, +/- misconceptions) are appended.

### Role

```
You are an educational designer specializing in Productive Failure (PF) exercises
for mathematics. Your job is to create problems that students attempt BEFORE
receiving direct instruction on a topic. The exercise should cause students to
generate plausible but incorrect solutions using only their prior knowledge —
setting up deeper understanding when the teacher later explains the canonical method.
```

### Output structure

```
Generate a PF exercise with exactly these sections:

1. TOPIC
   Grade level, math topic, and theme (e.g., "7th grade, ratios & proportions,
   basketball").

2. NARRATIVE
   A 1-paragraph real-world scenario with clear characters, a concrete situation,
   and stakes that make students care about solving the problem. Write at the
   reading level of the target grade.

3. DATA & CONSTRAINTS
   The numbers, facts, or conditions embedded in the narrative. Present these
   naturally within the story, NOT as a separate "given" list. The numbers must
   be deliberately chosen so that applying prior knowledge in the obvious way
   produces a WRONG answer.

4. TASK
   1-2 sentences at the end. Use verbs like "design", "invent", "figure out",
   "decide", "explain". NEVER use "solve for x", "calculate", or "what is the
   answer". The task must admit multiple solution approaches — different students
   should be able to attempt it in genuinely different ways.

5. CONSOLIDATION NOTE (for the teacher, NOT part of the student-facing exercise)
   The critical conceptual feature that students are expected to discover during
   the consolidation phase — i.e., what prior knowledge misses and why the
   canonical method works. Also list 2-3 expected student solution attempts
   (both correct and incorrect) that the teacher might see.
```

### Design rules

```
- The narrative must feel real and engaging, not like a textbook word problem.
- Data must be RIGGED: the most obvious approach (using only prior knowledge)
  must yield a plausible-looking but incorrect result.
- Do NOT state the critical conceptual feature in the student-facing part.
- The task wording must NOT hint at the correct method.
- For V3 (LLM-inferred misconceptions): if misconceptions are provided, design
  the data trap to specifically trigger at least one of them.
```

### Quality targets

Map each design element to the Rahimi et al. (2025) PF rubric (1-3 scale, target ≥ 2):

| Rubric dimension | Target | How to achieve it |
|---|---|---|
| APK — Activation of Prior Knowledge | ≥ 2 | The narrative and data should implicitly require at least two prerequisite concepts. |
| SSC — Sweet Spot Calibration | ≥ 2 | The obvious-but-wrong answer must look reasonable, not absurd. Students should feel "that makes sense" before realizing it's wrong. |
| AE — Affective Engagement | ≥ 2 | Use relatable characters, pop culture, or real-world scenarios for the target grade level. |
| MR — Multiple Representations | ≥ 2 | Include or suggest at least two formats (text + table, text + diagram reference, etc.). |
| OE — Open-Endedness | ≥ 2 | Multiple solution pathways must be genuinely possible. Never ask for a single numeric answer. |

### Example (from the literature, not generated)

See StratL few-shot examples: Kapur & Bielaczyc (2012) average speed problem,
Loibl & Rummel (2014) variance/consistency problem, Rahimi et al. (2025)
percentage change problem — all described in the anatomy section above.

## Topic selection

Following Rahimi et al. (2025): **single topic, controlled ablation**.

| What | Rahimi (2025) | Ours |
|---|---|---|
| Grade | 7th | 7th |
| Topic | Ratios & proportional relationships | Ratios & proportional relationships |
| Standard | CCSS `7.RP.A.2` | Same (or UK-equivalent from Eedi) |
| Prerequisites | Unit rates, rational number operations | Same |
| Theme | Sports / basketball | Same (or varied per generation) |
| N per version | 30 per model | 25 per version |

**Why n=25 per version (75 total):**

Rahimi et al. (2025) detected d=0.74 on SSC (comparing GPT-4o vs. Phi-3) with n=30 per model. Our manipulation — adding misconception data to the prompt — is expected to produce a smaller effect than switching between a state-of-the-art LLM and a 3.8B model. To detect a moderately large effect (d ≥ 0.79) with 80% power at α=0.05, n=25 per group is required:

$$n = \frac{2(z_{\alpha/2} + z_{\beta})^2}{d^2} = \frac{2(1.96 + 0.84)^2}{0.79^2} \approx 25$$

At n=25, we have 80% power for d ≥ 0.79 and ~65% power for Rahimi's observed d=0.74 — acceptable for a first exploration of misconception-grounded PF generation.

**Extension later**: add a second topic to test generalization — but start with one.

## Few-shot examples

Three examples from the 15 expert-authored PF exercises in `data/pf_exercises/`. Chosen for **pattern diversity** — each showcases a different PF structure so the LLM generalizes rather than copies one template.

### Selection rationale

| # | Source | Grade | PF Pattern | Why chosen |
|---|---|---|---|---|
| 1 | StratL: Consistency | 9 | **Data-table driven** — same mean + same range forces invention of a new measure | Classic PF structure. Clean, concise, GPT-4o optimized. |
| 2 | StratL: Country | 9 | **Ill-structured** — non-integer division creates the fairness trap | Different structure: open-ended, real-world political context. |
| 3 | Kapur: Average Speed | 7 | **Narrative-heavy** — dialogue-driven story, non-additive property | Canonical PF exercise. Matches target grade (7th). Storytelling as PF vehicle. |

Together: three distinct ways to rig data — (1) hidden mathematical equality, (2) non-integer fairness constraint, (3) non-additive physical property.

Each example follows the output structure (Topic → Narrative → Data & Constraints → Task) but **omits the Consolidation Note** — the original papers did not include teacher-facing notes, so we don't fabricate them. The LLM learns to generate the note from the system prompt instructions alone.

### Example 1/3 — StratL: Consistency

**TOPIC:** 9th grade, variance & consistency, Premier League soccer

**NARRATIVE:** The Premier League Federation must decide which of three players — Mike Arwen, Dave Backhand, or Ivan Right — should receive "The Most Consistent Player for the Past 5 Years" award. They've agreed to decide mathematically. They need your help.

**DATA & CONSTRAINTS:**

| Year | Mike | Dave | Ivan |
|------|------|------|------|
| 2019 | 13 | 12 | 14 |
| 2020 | 12 | 14 | 10 |
| 2021 | 15 | 16 | 18 |
| 2022 | 17 | 15 | 18 |
| 2023 | 13 | 13 | 15 |

All three players have the SAME mean (14). All three have the SAME range. Yet their patterns differ. A student using only mean or range concludes all three are equally consistent — which is wrong.

**TASK:** Design as many different measures of consistency as you can. Each measure must use all data points. Explain why each works and which player is most consistent.

---

### Example 2/3 — StratL: Country (Apportionment)

**TOPIC:** 9th grade, proportional reasoning & fair division, political representation

**NARRATIVE:** A new country has been founded, split into six states: A–F. Each has a different population. The country must allocate 250 legislative seats so every state receives fair representation. Unfair allocation means unequal voting power.

**DATA & CONSTRAINTS:**

| State | Population |
|-------|-----------|
| A | 1,646,000 |
| B | 6,936,000 |
| C | 154,000 |
| D | 2,091,000 |
| E | 685,000 |
| F | 988,000 |

Total: 12,500,000 people, 250 seats. Seats are whole numbers — can't be split. Simple rounding over- or under-allocates.

**TASK:** How many seats per state for fair representation? Show your work and justify your method.

---

### Example 3/3 — Kapur & Bielaczyc (2012): Average Speed

**TOPIC:** 7th grade, average speed, Singapore Idol audition

**NARRATIVE:** Hady and Jasmine bike together to the Singapore Idol auditions, 8.75 km away. After 25 minutes (3.75 km), Jasmine's tire flats. They have only Hady's bike. Their plan: Jasmine rides ahead, leaves the bike, walks. Hady walks until he reaches the bike, then rides.

**DATA & CONSTRAINTS:** Remaining: 5 km. Jasmine walks ~30 m/min, Hady walks ~50 m/min. Jasmine bikes 150 m/min, Hady bikes 200 m/min. Biking speed ratio (4:3) ≠ walking speed ratio (5:3). Using simple speed ratios to partition distance fails — average speed is total distance / total time, NOT the arithmetic mean of speeds.

**TASK:** How far should Jasmine ride before leaving the bike so both arrive at the same time? Show your reasoning.

## Evaluation plan

### Approach

LLM-as-judge with per-dimension scoring (Zheng et al., 2023). One prompt, one LLM call per exercise, all 5 dimensions scored together. Same pipeline as Phase 1a.

### Rubric

Rahimi et al. (2025) — 5 dimensions, 1–3 scale each, total 5–15.

| Dimension | Score 1 (Low) | Score 2 (Moderate) | Score 3 (High) |
|---|---|---|---|
| APK | No prerequisite concepts | One prerequisite concept | Two or more interacting prerequisites |
| SSC | Too easy or too hard | Moderate challenge, partial progress | Prior knowledge produces plausible-but-WRONG answer |
| AE | Abstract, no relatable context | Somewhat engaging, real-world scenario | Highly engaging, connects to student interests |
| MR | Text-only, single format | Two representation types | Three or more (table, diagram, etc.) |
| OE | Single answer, single path | Some flexibility | Multiple pathways; "design", "invent", "figure out" |

### Measurement quality (following Rahimi et al. 2025)

| Safeguard | Source | How |
|---|---|---|
| Blinding | Rahimi et al. (2025) | Judge sees exercise text only; no condition label (V1/V2/V3) |
| Deterministic | Zheng et al. (2023) | Temperature = 0; same exercise → same score |
| Shuffled order | Our pipeline | All exercises pooled and shuffled before scoring |
| Justification per dimension | Our pipeline | Each score requires a 1-sentence reason for debuggability |

### Judge prompt

Based on the Rahimi et al. (2025) rubric. Output format from our validated Phase 1a pipeline.

```
You are an expert evaluator of Productive Failure (PF) exercises for mathematics.
Score the exercise below on 5 dimensions using a 3-point scale.

1. Activation of Prior Knowledge (APK)
   1 = No prerequisite concepts required
   2 = One prerequisite concept
   3 = Two or more prerequisite concepts that interact

2. Sweet Spot Calibration (SSC)
   1 = Too easy or too hard; prior knowledge either trivially solves or cannot begin
   2 = Moderate challenge; prior knowledge produces partial progress
   3 = Well-calibrated: prior knowledge produces a plausible-looking but WRONG answer

3. Affective Engagement (AE)
   1 = Abstract or textbook-like; no relatable context
   2 = Somewhat engaging; real-world scenario present
   3 = Highly engaging; characters, stakes, and scenario connect to student interests

4. Multiple Representations (MR)
   1 = Text-only; single format
   2 = Two representation types (e.g., narrative + table)
   3 = Three or more types (e.g., narrative + table + diagram reference)

5. Open-Endedness (OE)
   1 = Single answer, single path; "solve for X"
   2 = Some flexibility; limited interpretation
   3 = Multiple genuine solution pathways; uses "design", "invent", "figure out"

OUTPUT:
APK: [1-3] — [justification]
SSC: [1-3] — [justification]
AE: [1-3] — [justification]
MR: [1-3] — [justification]
OE: [1-3] — [justification]
TOTAL: [sum]/15
```

### Analysis

Per exercise: 5 dimension scores + total. Per version (V1/V2/V3): mean ± SD per dimension, mean total. Between versions: Cohen's d per dimension.

### Human calibration

Spot-check ~20 exercises (~7 per version, randomly sampled). Score using the same rubric, blind to condition and LLM scores. Compute Pearson r between human and LLM scores.

Following Rahimi et al. (2025): r > 0.7 indicates acceptable reliability.

**Why n=20:** Pearson r has a wide confidence interval at small n. Fisher's z-transformation gives:

$$z = \frac{1}{2}\ln\!\left(\frac{1+r}{1-r}\right), \quad SE(z) = \frac{1}{\sqrt{n-3}}$$

$$95\%\ \text{CI for r:} \quad \tanh\!\big(z \pm 1.96 \cdot SE(z)\big)$$

| n | Observed r | 95% CI | Lower bound > 0.7? |
|---|---|---|---|
| 10 | 0.85 | [0.48, 0.96] | No — truth could be 0.48 |
| 15 | 0.85 | [0.58, 0.94] | No |
| **20** | **0.85** | **[0.65, 0.94]** | **Borderline — clears with observed r ≥ 0.88** |
| 25 | 0.85 | [0.68, 0.93] | Borderline |
| 30 | 0.85 | [0.71, 0.93] | Yes |

At n=20 with observed r ≥ 0.85, the 95% CI lower bound is 0.65 — close to the 0.7 threshold. With observed r ≥ 0.88, the lower bound clears 0.7. Given our validated Phase 1a judge pipeline and a well-specified rubric, r in the 0.85–0.90 range is a realistic expectation. n=20 is the smallest sample that gives a defensible confidence interval.

## Pipeline overview

### What happens end-to-end

```
┌─────────────────────────────────────────────────┐
│                 SYSTEM PROMPT                     │
│  PF definition + rubric + output structure +      │
│  design rules + 3 few-shot examples               │
│  (same for all versions, sent once)               │
└────────────────────┬────────────────────────────┘
                     │
         ┌───────────┼───────────┐
         ▼           ▼           ▼
       V1          V2          V3
    (Blind)    (Eedi)    (LLM-inferred)
         │           │           │
         ▼           ▼           ▼
┌───────────┐ ┌───────────┐ ┌───────────┐
│ User      │ │ User      │ │ User      │
│ prompt:   │ │ prompt:   │ │ prompt:   │
│ "7th      │ │ "7th      │ │ "7th      │
│ grade,    │ │ grade,    │ │ grade,    │
│ ratios,   │ │ ratios,   │ │ ratios,   │
│ basketball│ │ basketball│ │ basketball│
│ "         │ │           │ │           │
│           │ │ Misconcep-│ │ [LLM first │
│           │ │ tions:    │ │  infers    │
│           │ │ - mixes   │ │  misconcep-│
│           │ │   up sides│ │  tions,    │
│           │ │ - shares  │ │  then      │
│           │ │   equally │ │  generates]│
│           │ │ - treats  │ │            │
│           │ │   number  │ │            │
│           │ │   as total│ │            │
│           │ │ - ...     │ │            │
└─────┬─────┘ └─────┬─────┘ └─────┬─────┘
      │             │             │
      ▼             ▼             ▼
    25 PF        25 PF        25 PF
  exercises     exercises     exercises
      │             │             │
      └─────────────┼─────────────┘
                    │
                    ▼
          ┌─────────────────┐
          │   BLIND JUDGE   │
          │  (all exercises │
          │   pooled +      │
          │   shuffled)     │
          │  APK SSC AE     │
          │  MR OE → /15   │
          └────────┬────────┘
                   │
                   ▼
          ┌─────────────────┐
          │    ANALYSIS     │
          │  per version:   │
          │  mean ± SD      │
          │  Cohen's d      │
          └─────────────────┘
```

### Concrete example — same topic, three versions

**V1 (Blind):** generator receives only topic + theme. Must figure out the trap on its own.

```
→ LLM receives:
  "Generate a PF exercise for 7th grade, ratios & proportions, basketball theme."

← LLM outputs:
  TOPIC: 7th grade, ratios & proportions, basketball
  NARRATIVE: The Hawks need to mix paint for their new court. The instructions
  say to mix red and blue paint in a 3:5 ratio. Coach has 24 liters of red paint...
  DATA: 24L red, target ratio 3:5. Coach is tempted to just add 24 × 5/3 = 40L
  blue — but that's wrong because... [LLM must figure out the trap]
  TASK: Design a method to find the correct amount of blue paint. Show why the
  obvious approach fails.
  CONSOLIDATION NOTE: ... [LLM infers critical conceptual features]
```

**V2 (Eedi-grounded):** same prompt + Eedi misconception list. Generator is told exactly what students get wrong.

```
→ LLM receives:
  "Generate a PF exercise for 7th grade, ratios & proportions, basketball theme.
  Common student misconceptions about this topic:
  - Mixes up sides of a ratio
  - Divides total amount by each side of the ratio instead of sum of parts
  - Thinks the number in the ratio is the total
  - Shares equally instead of in given ratio
  - Thinks a difference of one part means quantities differ by one unit"

← LLM outputs:
  [Same format, but the data trap is specifically designed to trigger "divides
   total by each side instead of sum of parts" — e.g., the total happens to be
   divisible by both ratio numbers, making that error look correct.]
```

**V3 (LLM-inferred):** generator first asks itself what misconceptions exist, then uses its own answer.

```
→ LLM receives (step 1):
  "Students are about to learn ratios & proportions (7th grade). List 4-5 common
  misconceptions students typically have about this topic."

← LLM outputs:
  - Applying additive reasoning to multiplicative ratio relationships
  - Confusing part-to-part with part-to-whole comparisons
  - Thinking ratios can be simplified by subtracting
  - ... [LLM's own list]

→ LLM receives (step 2):
  "Now, using these misconceptions, generate a PF exercise for 7th grade, ratios
  & proportions, basketball theme."

← LLM outputs:
  [PF exercise informed by LLM's self-generated misconception list]
```

### Code structure (reusing Phase 1a)

```
src/generation/pf_generator.py   ← wraps LLM client, assembles prompts
prompts/pf_generation.yaml       ← system prompt + few-shot + user templates
src/evaluation/llm_judge.py      ← PF rubric judge (same pattern as Phase 1a)
scripts/run_phase1b_pf.py        ← orchestration: generate → judge → analyze
```

Same `llm_client.py`, same parallel execution (ThreadPool), same CSV output format.